<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Waterloo-Park-LiDAR-Tree-Detection-Pipeline
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A Machine Learning Exploration of LiDAR Classification
</h2>
</div>

##### Version Number: 1.0
---
### Contents  
---
### Notes
---
### Inputs
---
### Outputs  
---
### User Created Dependencies  
---
### Third Party Dependencies

In [142]:
import pandas as pd
import numpy as np
import laspy
import rasterio
from rasterio.transform import rowcol

### Load LAS file

In [ ]:
# Load the LAS/LAZ file
las = laspy.read("data/clipped_point_cloud.las")

cloud = pd.DataFrame({
    "X": np.array(las.x),
    "Y": np.array(las.y),
    "Z": np.array(las.z),
    "intensity": np.array(las.intensity),
    "classification": np.array(las.classification)
})

point_ids = np.arange(len(cloud))

### Load Orthophoto Raster

In [172]:
with rasterio.open("data/clipped_ortho.tif") as src:
    transform = src.transform
    bounds = src.bounds
    crs = src.crs
    
    x_min = bounds.left
    y_min = bounds.bottom
    x_max = bounds.right
    y_max = bounds.top

    width = src.width
    height = src.height

    red_band = src.read(1)  # Red
    green_band = src.read(2)  # Green
    blue_band = src.read(3)  # Blue

Isolate Ground Points

In [192]:
ground = cloud[cloud.classification == 2]

In [193]:
gx = ground.X
gy = ground.Y
gz = ground.Z

Group ground points in grid of orthophoto

In [ ]:
ground = cloud[cloud.classification == 2]

rows_idx, cols_idx = rasterio.transform.rowcol(
    transform,
    ground.X,
    ground.Y
)

## eliminate points outside bounds
mask = (
    (rows_idx >= 0) & (rows_idx < height) &
    (cols_idx >= 0) & (cols_idx < width)
)

ground_in_ortho = ground[mask]

In [ ]:
gr = pd.DataFrame({
    "row": rows_idx,
    "col": cols_idx,
    "z": ground.Z.values
})

In [ ]:
## Find mean elevation in each individual grid
mean_ground_grid = gr.groupby(["row", "col"])["z"].mean().reset_index()

Filter only unclassified points

In [ ]:
## use unclassified points only
trees = cloud[cloud.classification == 1]

Group unclassified points by position in orthophoto raster

In [ ]:
trees_rows_idx, trees_cols_idx = rasterio.transform.rowcol(
    transform,
    trees.X,
    trees.Y
)

In [ ]:
trees_df = pd.DataFrame({
    "row": trees_rows_idx,
    "col": trees_cols_idx,
    "z": trees.Z.values
})

Calculate Height Above Ground

In [ ]:
merged = trees_df.merge(
    mean_ground_grid,
    on=["row", "col"],
    how="left"
)

merged["HAG"] = merged["z"] - merged["ground_mean"]

- Identify individual canopy
- limit x,y to approximate canopy dimensions
- take vertical slices at DBH
- look for consistent gaps (underbrush should not contain a consistent gap) no idea how i would do this

CHM = generate_canopy_height_model(point_cloud)
canopy_peaks = find_local_maxima(CHM)

for each peak in canopy_peaks:
    canopy_region = watershed_segment(CHM, seed=peak)
    store canopy_region


for each canopy_region:
    canopy_points = all_points_within(canopy_region)

    centroid = mean(canopy_points.x), mean(canopy_points.y)
    footprint = convex_hull(canopy_points.xy)

    store (centroid, footprint)


for each canopy:
    radius = estimate_radius(footprint)
    cylinder_points = points_within_radius(point_cloud, centroid, radius)

    store cylinder_points


for each canopy:
    bins = create_vertical_bins(z_min, z_max, bin_height=0.25m)

    for each bin:
        density[bin] = count_points_in_bin(cylinder_points, bin)

    smoothed = smooth(density, window=3)

    gap = find_largest_contiguous_interval(smoothed < threshold)

    if gap.height < min_gap_height:
        mark canopy as shrub_or_invalid
        continue


for each valid canopy:
    dbh_slice = filter_points_by_z(cylinder_points, 1.3m, 1.5m)

    if dbh_slice is empty:
        mark canopy as no_trunk_detected
        continue


best_cluster = None
best_fit_error = INF

for each cluster in clusters:
    circle = fit_circle_least_squares(cluster.xy)

    if circle.residual < best_fit_error AND circle.radius < max_expected_radius:
        best_cluster = cluster
        best_circle = circle
        best_fit_error = circle.residual

if best_cluster is None:
    mark canopy as no_trunk_detected
    continue

trunk_center = best_circle.center
trunk_radius = best_circle.radius


vertical_cylinder = points_within_radius(point_cloud, trunk_center, 0.4m)

bins = create_vertical_bins(z_min, z_max, 0.25m)

for each bin:
    count[bin] = number_of_points(vertical_cylinder in bin)

continuity_score = count_nonempty_bins_in_order(count)

if continuity_score < threshold:
    mark canopy as low_confidence_trunk
else:
    mark canopy as confirmed_trunk


for each canopy:
    if trunk_center exists:
        for each point in canopy_region:
            assign point to trunk_center

        refine canopy boundary using trunk_center
